### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [30]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [31]:
# Read all the pdf's inside the directory 

def process_all_pdf(pdf_directory):
    """Process all the PDF files"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f' loaded {len(documents)} pages')

        except Exception as e:
            print(f' Error:{e}')

    print(f'\nTotal documents loaded: {len(all_documents)}')
    return all_documents 

all_pdf_documents = process_all_pdf("../data")

Found 3 PDF files to process

Processing: AI Agents.pdf
 loaded 30 pages

Processing: Claude-Code.pdf
 loaded 23 pages

Processing: Anthropic.pdf
 loaded 2 pages

Total documents loaded: 55


In [32]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content="Building Effective AI Agents: \nArchitecture Patterns and \nImplementation Frameworks\nAccelerate your enterprise AI transformation \nwith proven strategies from Anthropic's \ncustomers and internal teams."),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 1, 'page_label': '2', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content='Content s\nExecutive summary 4\nCommon 

In [33]:
### Text Splitting into chunks 

def split_documents(documents,chunk_size = 1000, chunk_overlap= 200):
    """Splitting documents into smaller chunks for RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')

    # Example of a Chunk 
    if split_docs:
        print(f'\nExample Chunk:')
        print(f'\ncontent: {split_docs[0].page_content[:200]}...')
        print(f'\nMetadata: {split_docs[0].metadata}')

    return split_docs 


In [34]:
chunks = split_documents(all_pdf_documents)
chunks

Split 55 documents into 234 chunks

Example Chunk:

content: Building Effective AI Agents: 
Architecture Patterns and 
Implementation Frameworks
Accelerate your enterprise AI transformation 
with proven strategies from Anthropic's 
customers and internal teams....

Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content="Building Effective AI Agents: \nArchitecture Patterns and \nImplementation Frameworks\nAccelerate your enterprise AI transformation \nwith proven strategies from Anthropic's \ncustomers and internal teams."),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 1, 'page_label': '2', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content='Content s\nExecutive summary 4\nCommon 

### Embedding and vector store DB 

In [35]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-l6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:    
            print(f'Error Loading model {self.model_name}: {e}')
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f'Generating embeddings for {len(texts)} texts....')
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f'Generated embeddings with shape: {embeddings.shape}')
        return embeddings 
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-l6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6921.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


/var/folders/x2/smsrx5p17hd6dqglw21wmzmm0000gn/T/ipykernel_14973/3489931333.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store 

In [53]:
class VectorStore:
    """Manages embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f'vector store intialized. collection: {self.collection_name}')
            print(f'Existing documents in collection: {self.collection.count()}')

        except Exception as e:
            print(f'Error intializing vector store: {e}')
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        if len(documents) != len(embeddings):
            raise ValueError(" Number of documents must match embeddings")
        
        print(f'Adding {len(documents)} documents to vector store')

        # Data for Chromadb 
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            try: 
                self.collection.add(
                    ids=ids,
                    embeddings = embeddings_list,
                    metadatas = metadatas,
                    documents = documents_text
                ) 
                print(f'Successfully added {len(documents)} documents to vector store')
                print(f'Total documents in collection: {self.collection.count()}')

            except Exception as e:
                print(f'Error adding docuemnts to vector store: {e}')
                raise 

vector_store = VectorStore()
vector_store

vector store intialized. collection: pdf_documents
Existing documents in collection: 0


In [47]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content="Building Effective AI Agents: \nArchitecture Patterns and \nImplementation Frameworks\nAccelerate your enterprise AI transformation \nwith proven strategies from Anthropic's \ncustomers and internal teams."),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.0 (Macintosh)', 'creationdate': '2025-12-02T15:07:20-07:00', 'moddate': '2025-12-02T15:07:21-07:00', 'trapped': '/False', 'source': '../data/text_files/pdf/AI Agents.pdf', 'total_pages': 30, 'page': 1, 'page_label': '2', 'source_file': 'AI Agents.pdf', 'file_type': 'pdf'}, page_content='Content s\nExecutive summary 4\nCommon 

In [48]:
# convert text to embeddings 
texts = [doc.page_content for doc in chunks]
texts

["Building Effective AI Agents: \nArchitecture Patterns and \nImplementation Frameworks\nAccelerate your enterprise AI transformation \nwith proven strategies from Anthropic's \ncustomers and internal teams.",
 'Content s\nExecutive summary 4\nCommon use cases and applications f or AI agents 7\nCommon architecture patterns 10\nLooking f orward: the future of building AI agents 27\nNext steps 29\n2',
 'Chapter 1\nExecutive summary\n3',
 "Chapter 1\nExecutive summary\nGenerative AI answers questions. AI agents solve \nproblems. \nFor companies across industries, agents offer the potential for scaling operations \nin ways current automation could never deliver: open-ended problem-solving, \ndynamic decision-making, and complex multistep processes where the path \nforward isn't predetermined. \nOrganizations are seeing significant results from autonomous agents in \nproduction. For example, Coinbase, the leading cryptocurrency exchange \nmanaging $226 billion in quarterly trading volume, b

In [54]:
# Generate Embeddings 

embeddings = embedding_manager.generate_embeddings(texts)

# store in vector database 

vector_store.add_documents(chunks, embeddings)

Generating embeddings for 234 texts....


Batches: 100%|██████████| 8/8 [00:01<00:00,  6.74it/s]


Generated embeddings with shape: (234, 384)
Adding 234 documents to vector store
Successfully added 234 documents to vector store
Total documents in collection: 1
Successfully added 234 documents to vector store
Total documents in collection: 2
Successfully added 234 documents to vector store
Total documents in collection: 3
Successfully added 234 documents to vector store
Total documents in collection: 4
Successfully added 234 documents to vector store
Total documents in collection: 5
Successfully added 234 documents to vector store
Total documents in collection: 6
Successfully added 234 documents to vector store
Total documents in collection: 7
Successfully added 234 documents to vector store
Total documents in collection: 8
Successfully added 234 documents to vector store
Total documents in collection: 9
Successfully added 234 documents to vector store
Total documents in collection: 10
Successfully added 234 documents to vector store
Total documents in collection: 11
Successfully ad